<a href="https://colab.research.google.com/github/donleaveher/plasticity-placement/blob/agent%2Fadd-lora-evaluation/notebooks/p0d_lora_locus/p0d_band_scan_colab.ipynb" target="_parent"><img
src="https://colab.research.google.com/assets/colab-badge.svg"
alt="Open In Colab"/></a>

# P0-D LoRA Layer Locus — Band Scan

Standalone notebook for **band_scan only**. Requires the complete verified P0-C v4 Confirmatory manifest.
P0-C inputs are read-only; all P0-D outputs use the independent
`plasticity-p0d/lora-locus/v1` Drive namespace.

## 1. Configuration

Select a GPU runtime. Point the P0-C attempt labels at the verified
Confirmatory run. Increment only this P0-D stage attempt after an immutable
failure or environment change.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import json
import os
import re
import shlex
import subprocess
import sys
import time
from collections import Counter
from datetime import datetime, timezone
from hashlib import sha256
from pathlib import Path

from IPython.display import display

REPO_URL = 'https://github.com/donleaveher/plasticity-placement.git'
BRANCH = 'agent/add-lora-evaluation'
REQUESTED_CODE_REVISION = None
REPO_DIR = Path('/content/plasticity-placement')

P0C_PIPELINE_ATTEMPT = 'pipeline-a1'
P0C_CONFIRMATORY_ATTEMPT = 'a1'
P0C_PIPELINE_ROOT = (
    Path('/content/drive/MyDrive/plasticity-p0c/v4/pipelines')
    / P0C_PIPELINE_ATTEMPT
)

P0D_PIPELINE_ATTEMPT = 'pipeline-a1'
BAND_SCAN_ATTEMPT = 'a1'

P0D_PIPELINE_ROOT = (
    Path('/content/drive/MyDrive/plasticity-p0d/lora-locus/v1/pipelines')
    / P0D_PIPELINE_ATTEMPT
)
NARROW_CONDITIONS_CONFIG = (
    P0D_PIPELINE_ROOT / 'frozen-narrow-conditions.json'
)

RUN_BAND_SCAN = False

for name, value in {
    'P0C_PIPELINE_ATTEMPT': P0C_PIPELINE_ATTEMPT,
    'P0C_CONFIRMATORY_ATTEMPT': P0C_CONFIRMATORY_ATTEMPT,
    'P0D_PIPELINE_ATTEMPT': P0D_PIPELINE_ATTEMPT,
    'BAND_SCAN_ATTEMPT': BAND_SCAN_ATTEMPT,

}.items():
    if not re.fullmatch(r'[A-Za-z0-9._-]+', value):
        raise ValueError(f'{name} contains unsafe path characters: {value!r}')

## 2. Checkout and install the frozen P0-D code

In [ ]:
subprocess.run(['nvidia-smi'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'], check=True)

if REPO_DIR.exists() and not (REPO_DIR / '.git').exists():
    raise RuntimeError(f'{REPO_DIR} exists but is not a Git repository')
if not REPO_DIR.exists():
    subprocess.run(
        ['git', 'clone', '--branch', BRANCH, REPO_URL, str(REPO_DIR)],
        check=True,
    )

dirty = subprocess.run(
    ['git', '-C', str(REPO_DIR), 'status', '--porcelain'],
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()
if dirty:
    raise RuntimeError(
        'The Colab checkout contains local changes; use a fresh runtime.\n' + dirty
    )

P0D_PIPELINE_ROOT.mkdir(parents=True, exist_ok=True)
CODE_REVISION_LOCK = P0D_PIPELINE_ROOT / 'code-revision.txt'
locked_revision = (
    CODE_REVISION_LOCK.read_text().strip()
    if CODE_REVISION_LOCK.exists()
    else None
)
subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', BRANCH], check=True)
revision_ref = REQUESTED_CODE_REVISION or locked_revision or f'origin/{BRANCH}'
CODE_REVISION = subprocess.run(
    ['git', '-C', str(REPO_DIR), 'rev-parse', f'{revision_ref}^{{commit}}'],
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()
if locked_revision and CODE_REVISION != locked_revision:
    raise RuntimeError(
        f'P0-D code lock mismatch: {locked_revision} != {CODE_REVISION}. '
        'Use a new P0D_PIPELINE_ATTEMPT.'
    )
if not locked_revision:
    temporary = CODE_REVISION_LOCK.with_suffix('.txt.tmp')
    temporary.write_text(CODE_REVISION + '\n')
    temporary.replace(CODE_REVISION_LOCK)
subprocess.run(
    ['git', '-C', str(REPO_DIR), 'checkout', '--detach', CODE_REVISION],
    check=True,
)
subprocess.run(
    ['uv', 'sync', '--extra', 'train', '--extra', 'colab'],
    cwd=REPO_DIR,
    check=True,
)
print('Checked out P0-D code:', CODE_REVISION)

## 3. Resolve source provenance, stage directory, and recovery

In [ ]:
def run_json(command):
    completed = subprocess.run(
        command,
        cwd=REPO_DIR,
        check=False,
        capture_output=True,
        text=True,
    )
    if completed.returncode != 0:
        print(completed.stdout)
        print(completed.stderr)
        raise RuntimeError(
            f'Command failed ({completed.returncode}): {shlex.join(command)}'
        )
    lines = [line for line in completed.stdout.splitlines() if line.strip()]
    if not lines:
        raise RuntimeError(f'Command returned no JSON: {shlex.join(command)}')
    return json.loads(lines[-1])

source_matches = sorted(
    P0C_PIPELINE_ROOT.glob(
        'runs/*/confirmatory-'
        + P0C_CONFIRMATORY_ATTEMPT
        + '/manifest.json'
    )
)
if len(source_matches) != 1:
    raise RuntimeError(
        'Expected exactly one P0-C Confirmatory manifest under '
        f'{P0C_PIPELINE_ROOT}, found {source_matches}'
    )
SOURCE_P0C_MANIFEST = source_matches[0]
source_manifest = json.loads(SOURCE_P0C_MANIFEST.read_text())
source_lessons = source_manifest.get('selected_lessons', [])
source_units = source_manifest.get('units', {})
SOURCE_MODEL_REVISION = source_manifest.get('config', {}).get('model_revision')
if (
    source_manifest.get('config', {}).get('tier') != 'confirmatory'
    or len(source_lessons) != 24
    or len(source_units) != 72
    or set(source_manifest.get('base_arms', {}).values()) != {'verified'}
    or any(
        unit.get('state') != 'verified'
        or float(unit.get('rollback_exact_match_rate', 0.0)) != 1.0
        for unit in source_units.values()
    )
):
    raise RuntimeError('P0-C source manifest is not a complete verified Confirmatory run')
SOURCE_MANIFEST_HASH = sha256(SOURCE_P0C_MANIFEST.read_bytes()).hexdigest()
SELECTED_LESSON_HASH = sha256(
    json.dumps(source_lessons, separators=(',', ':')).encode()
).hexdigest()

environment_context = run_json(['uv', 'run', 'plasticity-p0d', 'environment'])
environment = environment_context['environment']
ENVIRONMENT_FINGERPRINT = environment_context['fingerprint']
CODE_HASH = environment['code_sha256']

CONDITIONS_CONFIG = None
CONDITIONS_HASH = 'frozen-band-scan-v1'

PROVENANCE_KEY = (
    f'code-{CODE_HASH[:10]}_source-{SOURCE_MANIFEST_HASH[:10]}'
)
PROVENANCE_ROOT = P0D_PIPELINE_ROOT / 'runs' / PROVENANCE_KEY
STAGE_DIR = PROVENANCE_ROOT / ('band_scan-' + BAND_SCAN_ATTEMPT)
LOG_DIR = PROVENANCE_ROOT / 'logs'
LOG_DIR.mkdir(parents=True, exist_ok=True)

frozen_context = {
    'schema_version': 1,
    'pipeline_version': 'p0d-lora-locus-v1',
    'code_sha256': CODE_HASH,
    'source_manifest_sha256': SOURCE_MANIFEST_HASH,
    'source_model_revision': SOURCE_MODEL_REVISION,
    'selected_lesson_sha256': SELECTED_LESSON_HASH,
    'stage': 'band_scan',
    'conditions_sha256': CONDITIONS_HASH,
}
context_path = STAGE_DIR / 'notebook_context.json'
STAGE_DIR.mkdir(parents=True, exist_ok=True)
if context_path.exists():
    if json.loads(context_path.read_text()) != frozen_context:
        raise RuntimeError(
            f'Frozen P0-D stage context mismatch: {context_path}'
        )
else:
    temporary = context_path.with_suffix('.json.tmp')
    temporary.write_text(
        json.dumps(frozen_context, indent=2, sort_keys=True) + '\n'
    )
    temporary.replace(context_path)

sessions_path = STAGE_DIR / 'source_sessions.json'
sessions = json.loads(sessions_path.read_text()) if sessions_path.exists() else []
sessions.append({
    'recorded_at': datetime.now(timezone.utc).isoformat(),
    'git_revision': CODE_REVISION,
    'branch': BRANCH,
    'stage': 'band_scan',
    'environment_fingerprint': ENVIRONMENT_FINGERPRINT,
    'environment': environment,
    'source_p0c_manifest': str(SOURCE_P0C_MANIFEST),
    'conditions_config': str(CONDITIONS_CONFIG) if CONDITIONS_CONFIG else None,
})
temporary = sessions_path.with_suffix('.json.tmp')
temporary.write_text(json.dumps(sessions, indent=2, sort_keys=True) + '\n')
temporary.replace(sessions_path)

def run_checked(label, command):
    timestamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
    safe_label = re.sub(r'[^A-Za-z0-9._-]+', '-', label)
    log_path = LOG_DIR / f'{timestamp}-{safe_label}.log'
    child_environment = {**os.environ, 'PYTHONUNBUFFERED': '1'}
    print(f'\n[{label}] $ {shlex.join(command)}')
    print(f'[{label}] log: {log_path}')
    started = time.monotonic()
    with log_path.open('w', encoding='utf-8') as log_file:
        process = subprocess.Popen(
            command,
            cwd=REPO_DIR,
            env=child_environment,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        assert process.stdout is not None
        try:
            for line in process.stdout:
                print(line, end='')
                log_file.write(line)
                log_file.flush()
        except KeyboardInterrupt:
            process.terminate()
            process.wait()
            raise
        return_code = process.wait()
    elapsed = time.monotonic() - started
    print(f'[{label}] exit={return_code} elapsed={elapsed / 60:.1f} min')
    if return_code != 0:
        raise RuntimeError(
            f'{label} failed with exit code {return_code}; log={log_path}'
        )

def p0d_command(action):
    if sha256(SOURCE_P0C_MANIFEST.read_bytes()).hexdigest() != SOURCE_MANIFEST_HASH:
        raise RuntimeError('P0-C source manifest changed after provenance freeze')
    if (
        CONDITIONS_CONFIG is not None
        and sha256(CONDITIONS_CONFIG.read_bytes()).hexdigest() != CONDITIONS_HASH
    ):
        raise RuntimeError('P0-D conditions config changed after provenance freeze')
    command = [
        'uv', 'run', 'plasticity-p0d', action,
        '--output', str(STAGE_DIR),
    ]
    if action in {'plan', 'run'}:
        command.extend([
            '--source-manifest', str(SOURCE_P0C_MANIFEST),
            '--stage', 'band_scan',
        ])
        if CONDITIONS_CONFIG is not None:
            command.extend(['--conditions-config', str(CONDITIONS_CONFIG)])
    return command

def manifest_status():
    path = STAGE_DIR / 'manifest.json'
    if not path.exists():
        return {'exists': False, 'stage_dir': str(STAGE_DIR)}
    manifest = json.loads(path.read_text())
    return {
        'exists': True,
        'run_id': manifest.get('run_id'),
        'stage_dir': str(STAGE_DIR),
        'lesson_count': len(manifest.get('selected_lessons', [])),
        'conditions': list(manifest.get('conditions', {})),
        'base_states': dict(Counter(manifest.get('base_arms', {}).values())),
        'unit_states': dict(Counter(
            unit.get('state') for unit in manifest.get('units', {}).values()
        )),
        'errors': manifest.get('errors', [])[-10:],
    }

def verify_manifest(expected_units):
    path = STAGE_DIR / 'manifest.json'
    if not path.exists():
        raise FileNotFoundError(f'Missing P0-D manifest: {path}')
    manifest = json.loads(path.read_text())
    units = manifest.get('units', {})
    if len(manifest.get('selected_lessons', [])) != 24:
        raise RuntimeError('P0-D manifest does not contain 24 lessons')
    if len(units) != expected_units:
        raise RuntimeError(
            f'Expected {expected_units} adapter units, got {len(units)}'
        )
    if set(manifest.get('base_arms', {}).values()) != {'verified'}:
        raise RuntimeError('P0-D base arms are incomplete')
    bad = {
        key: unit.get('state')
        for key, unit in units.items()
        if unit.get('state') != 'verified'
        or float(unit.get('rollback_exact_match_rate', 0.0)) != 1.0
    }
    if bad or manifest.get('errors'):
        raise RuntimeError(
            f'P0-D units invalid={bad}, errors={manifest.get("errors", [])[-5:]}'
        )
    return manifest

print(json.dumps({
    'code_revision': CODE_REVISION,
    'code_sha256': CODE_HASH,
    'source_manifest': str(SOURCE_P0C_MANIFEST),
    'source_manifest_sha256': SOURCE_MANIFEST_HASH,
    'source_model_revision': SOURCE_MODEL_REVISION,
    'selected_lesson_sha256': SELECTED_LESSON_HASH,
    'conditions_sha256': CONDITIONS_HASH,
    'environment_fingerprint': ENVIRONMENT_FINGERPRINT,
    'stage_dir': str(STAGE_DIR),
}, indent=2))

## 4. Run Band Scan

In [ ]:
plan = run_json(p0d_command('plan'))
if plan['expected_unit_count'] != 288:
    raise RuntimeError(f"Band scan expected 288 units, got {plan['expected_unit_count']}")
display(plan['config']['conditions'])

if RUN_BAND_SCAN:
    run_checked('p0d-band-scan-run', p0d_command('run'))
    verify_manifest(expected_units=288)
    run_checked(
        'p0d-band-scan-aggregate',
        [
            'uv', 'run', 'plasticity-p0d', 'aggregate',
            '--output', str(STAGE_DIR),
            '--bootstrap-samples', '10000',
        ],
    )

print(json.dumps(manifest_status(), ensure_ascii=False, indent=2))
summary_path = STAGE_DIR / 'results' / 'aggregate' / 'summary.json'
if summary_path.exists():
    summary = json.loads(summary_path.read_text())
    if summary['adapter_unit_count'] != 288 or summary['probe_row_count'] != 16224:
        raise RuntimeError('Unexpected P0-D band-scan result dimensions')
    display(summary['condition_summary'])
    display(summary['contrasts'])
    display(summary['gates'])
    print(
        'Next-stage candidate file:',
        STAGE_DIR / 'results' / 'aggregate' / 'next_stage_candidates.json',
    )

## Recovery policy

A normal disconnect can resume verified units in the same stage attempt when
code, source manifest, condition matrix, and environment match. Preserve any
attempt containing a `failed` unit and increment this stage's attempt label.
Never edit manifests, raw rows, adapters, aggregate files, or frozen
conditions in place.